In [1]:
from transformers import VisionEncoderDecoderModel, DonutProcessor
import torch
from PIL import Image

In [3]:
load_path = "/Users/yiding/personal_projects/ML/github_repo/donut/src/test/content_recognition/donut-artifacts"  

# load processor（including special tokens）
processor = DonutProcessor.from_pretrained(load_path)

# load model（include config and weights）
model = VisionEncoderDecoderModel.from_pretrained(load_path)

# ensure decoder embedding extention
model.decoder.resize_token_embeddings(len(processor.tokenizer))


model.eval()
device = torch.device("mps" if torch.cuda.is_available() else "cpu")
model.to(device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0-1): 2 x DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
           

In [ ]:
# load image
image = Image.open("/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/348cdffe48e6.jpg").convert("RGB")

# process image
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

# create decoder start token
decoder_input_ids = torch.full(
    (1, 1),
    model.config.decoder_start_token_id,
    dtype=torch.long,
    device=device,
)

# generate
with torch.no_grad():
    output = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=model.config.max_length,
        num_beams=4,
    )

# decode
prediction = processor.tokenizer.decode(output[0], skip_special_tokens=False)
print("output:\n", prediction)